# Sesión 8

+ Esta sesión trabajaremos con datos del Federal Reserve Bank of St. Louis

+ El sitio web es https://fred.stlouisfed.org/

+ Vamos a TOOLS -> FRED API

In [ ]:
#!pip install fredapi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from fredapi import Fred
import time

In [ ]:
mi_fred_key = 'pega_aqui_tu_key'

In [ ]:
objeto_fred = Fred(api_key = mi_fred_key)

In [ ]:
objeto_fred

In [ ]:
busqueda = objeto_fred.search('S&P')

In [ ]:
busqueda

In [ ]:
type(busqueda)

In [ ]:
busqueda.shape

In [ ]:
busqueda = objeto_fred.search('S&P', order_by='popularity')

In [ ]:
busqueda

+ Hasta el momento sólo hemos hecho búsquedas, no descargado datos aún

In [ ]:
# Descargo una serie específica
serie_sp500 = objeto_fred.get_series(series_id = 'SP500')

In [ ]:
serie_sp500

In [ ]:
type(serie_sp500)

In [ ]:
serie_sp500.plot(title='S&P 500')
plt.show()

+ Vamos a hacer otra búsqueda

+ Supongamos que nos interesa la tasa de desempleo histórica por estado

+ Empecemos buscando...

In [ ]:
otra_busqueda = objeto_fred.search('unemployment rate state')

In [ ]:
otra_busqueda

In [ ]:
# Podemos refinar nuestra búsqueda
otra_busqueda = objeto_fred.search('unemployment rate state', filter = ('frequency','Monthly'))

In [ ]:
otra_busqueda

In [ ]:
type(otra_busqueda)

+ El objeto `otra_busqueda` es un pandas dataframe

+ Podemos navegar dentro de él como ya sabemos

In [ ]:
otra_busqueda.query('seasonal_adjustment == "Seasonally Adjusted" and units == "Percent"')

In [ ]:
otra_busqueda.loc[otra_busqueda['title'].str.contains('Unemployment Rate')]

In [ ]:
# Pongamos todo junto
mi_busqueda = objeto_fred.search('unemployment rate state', filter = ('frequency','Monthly'))
mi_busqueda = mi_busqueda.query('seasonal_adjustment == "Seasonally Adjusted" and units == "Percent"')
mi_busqueda = mi_busqueda.loc[mi_busqueda['title'].str.contains('Unemployment Rate')]

In [ ]:
mi_busqueda

+ Hasta ahora hemos hecho búsquedas, no descargado datos.

+ Tenemos metadatos, pero no datos en sí.

+ Vamos a descargar todas estas series

In [ ]:
lista_descargas = []

for id in mi_busqueda.index:
    results = objeto_fred.get_series(id)
    results = results.to_frame(name=id)
    lista_descargas.append(results)
    time.sleep(0.1) # Para modular el ritmo de las consultas porque FRED te puede bloquear

+ En el objeto `lista_descargas` están todas las series que ya descargué.

In [ ]:
#lista_descargas

+ Vamos a mejor ponerlas en un pandas DataFrame

In [ ]:
df_descargas = pd.concat(lista_descargas, axis=1)

In [ ]:
df_descargas.head()

In [ ]:
cols_eliminar = []

for i in df_descargas:
    if len(i) > 4:
        cols_eliminar.append(i)

In [ ]:
cols_eliminar

In [ ]:
df_descargas = df_descargas.drop(columns = cols_eliminar)

In [ ]:
df_descargas

In [ ]:
mis_series_historicas = df_descargas.copy()

In [ ]:
mis_series_historicas

In [ ]:
mis_series_historicas = mis_series_historicas.dropna()

In [ ]:
mis_series_historicas

+ Los nombres de las columnas son un lío

+ Tengo que ponerle nombres legibles

In [ ]:
mi_busqueda['title']

In [ ]:
dic_id_con_letra = mi_busqueda['title'].str.replace('Unemployment Rate in ','').to_dict()

In [ ]:
dic_id_con_letra

In [ ]:
nombres_estados = [dic_id_con_letra[j] for j in mis_series_historicas.columns]
nombres_estados

In [ ]:
dicc_nombres = {key: value for key, value in dic_id_con_letra.items() if value in nombres_estados}
dicc_nombres

In [ ]:
mis_series_historicas = mis_series_historicas.rename(columns = dicc_nombres)

In [ ]:
mis_series_historicas

In [ ]:
px.line(mis_series_historicas)

In [ ]:
datos_2025 = mis_series_historicas.loc[mis_series_historicas.index == '2025-05-01'].T.sort_values('2025-05-01')
datos_2025

In [ ]:
ax = datos_2025.plot(kind='barh', title='Tasa de desempleo por estado, mayo 2025', width=0.7, figsize=(8, 12))
ax.legend().remove()
ax.set_xlabel('% desempleo')
plt.show()

+ Hagamos otra búsqueda relacionada

+ La *Labor Force Participation Rate* (Tasa de Participación Laboral) es el porcentaje de la población en edad de trabajar (generalmente 15 o 16 años en adelante) que está empleada o busca trabajo activamente. Mide la oferta de mano de obra disponible y excluye a estudiantes, jubilados o personas que no buscan empleo.

In [ ]:
busqueda_part = objeto_fred.search('participation rate state', filter=('frequency','Monthly'))
busqueda_part = busqueda_part.query('seasonal_adjustment == "Seasonally Adjusted" and units == "Percent"')

In [ ]:
busqueda_part

+ Como antes, sólo hemos hecho búsquedas, no hemos descargado nada aún.

In [ ]:
resultados_part = []

for id in busqueda_part.index:
    results = objeto_fred.get_series(id)
    results = results.to_frame(name = id)
    resultados_part.append(results)
    time.sleep(0.1) # Para modular el ritmo de las consultas porque FRED te puede bloquear

In [ ]:
df_descargas_part = pd.concat(resultados_part, axis=1)

In [ ]:
df_descargas_part

In [ ]:
cols_eliminar = []

for i in df_descargas_part:
    if len(i) > 7:
        cols_eliminar.append(i)

In [ ]:
cols_eliminar

In [ ]:
df_descargas_part = df_descargas_part.drop(columns = cols_eliminar)

In [ ]:
df_descargas_part

In [ ]:
mis_otras_series_historicas = df_descargas_part.copy()

In [ ]:
mis_otras_series_historicas

In [ ]:
mis_otras_series_historicas = mis_otras_series_historicas.dropna()

In [ ]:
mis_otras_series_historicas

+ De nuevo, los nombres de las columnas son un lío

+ Tengo que ponerle nombres legibles

In [ ]:
busqueda_part#['title']

In [ ]:
dic_id_con_letra = busqueda_part['title'].str.replace('Labor Force Participation Rate for ','').to_dict()
dic_id_con_letra

In [ ]:
nombres_estados = [dic_id_con_letra[j] for j in mis_otras_series_historicas.columns]
nombres_estados

In [ ]:
dicc_nombres = {key: value for key, value in dic_id_con_letra.items() if value in nombres_estados}
dicc_nombres

In [ ]:
mis_otras_series_historicas = mis_otras_series_historicas.rename(columns = dicc_nombres)

In [ ]:
mis_otras_series_historicas

In [ ]:
mis_otras_series_historicas.columns

In [ ]:
mis_otras_series_historicas = mis_otras_series_historicas.drop(columns = 'Labor Force Participation Rate')

In [ ]:
mis_otras_series_historicas

In [ ]:
px.line(mis_otras_series_historicas)

In [ ]:
datos_part_2025 = mis_otras_series_historicas.loc[mis_otras_series_historicas.index == '2025-05-01'].T.sort_values('2025-05-01')
datos_part_2025

In [ ]:
ax = datos_part_2025.plot(kind='barh', title='Participación por estado, mayo 2025', width=0.7, figsize=(8, 12))
ax.legend().remove()
ax.set_xlabel('% participación')
plt.show()

¿Cómo los juntamos?

In [ ]:
df_desempleo = mis_series_historicas.copy()
df_part = mis_otras_series_historicas.copy()

In [ ]:
df_part.columns = (df_part.columns
                   .str.strip()
                   .str.lower()
                   .str.replace(" ", "_")
                   .str.replace("[()€$]", "", regex = True))

In [ ]:
df_desempleo.columns = (df_desempleo.columns
                        .str.strip()
                        .str.lower()
                        .str.replace(" ", "_")
                        .str.replace("[()€$]", "", regex = True))

In [ ]:
df_desempleo.head()

In [ ]:
df_part.head()

In [ ]:
df_desempleo['fecha'] = df_desempleo.index
df_part['fecha'] = df_part.index

In [ ]:
df_desempleo.head()

In [ ]:
df_part.head()

In [ ]:
df_part_largo = pd.melt(df_part, id_vars=['fecha'], var_name = 'estado', value_name = 'labor_part')
df_desempleo_largo = pd.melt(df_desempleo, id_vars=['fecha'], var_name = 'estado', value_name = 'desempleo')

In [ ]:
df_desempleo_largo.head()

In [ ]:
df_part_largo.head()

In [ ]:
df_junto = pd.merge(df_part_largo, df_desempleo_largo, on=['fecha', 'estado'], how='inner')

In [ ]:
df_junto

In [ ]:
df_junto = df_junto.set_index('fecha')

In [ ]:
df_junto

In [ ]:
df_junto.plot()
plt.show()

In [ ]:
df_junto.query('estado == "georgia"').plot()
plt.show()

In [ ]:
df_junto.query('estado == "district_of_columbia"').plot()
plt.show()

# Otra forma un poco más dolorosa

In [ ]:
import requests

In [ ]:
base_url = 'https://api.stlouisfed.org/fred/'
obs_endpoint = 'series/observations'

series_id = 'CPIAUCSL'
start_date = '2000-01-01'
end_date = '2023-06-30'

obs_params = {
    'series_id': series_id,
    'api_key': mi_fred_key,
    'file_type': 'json',
    'observation_start': start_date,
    'observation_end': end_date
}

# Hace la solicitud a la FRED API
response = requests.get(base_url + obs_endpoint, params = obs_params)

In [ ]:
response

In [ ]:
res_data = response.json()
#res_data

In [ ]:
df_descarga = pd.DataFrame(res_data['observations'])

In [ ]:
df_descarga.head()

In [ ]:
df_descarga = df_descarga[['date','value']]

In [ ]:
df_descarga = df_descarga.set_index('date')

In [ ]:
df_descarga.head()

In [ ]:
df_descarga.info()

In [ ]:
df_descarga['value'] = df_descarga['value'].astype(float)

In [ ]:
df_descarga.head()

In [ ]:
df_descarga.plot()
plt.show()

In [ ]:
busqueda = objeto_fred.search('CPIAUCSL')

In [ ]:
busqueda

In [ ]:
cat_id = '32455'
cat_srs_endpoint = 'category/series'
cat_srs_params = {
    'api_key': mi_fred_key,
    'file_type': 'json',
    'category_id': cat_id,
}

response = requests.get(base_url + cat_srs_endpoint, params=cat_srs_params)

In [ ]:
response

In [ ]:
res_data = response.json()
#res_data

In [ ]:
df_descarga = pd.DataFrame(res_data['seriess'])

In [ ]:
df_descarga